WhatNet - EMNIST-Balanced

In [ ]:
#  EMNIST Balanced: 131,600 chars, 47 balanced classes
#  Digits 0-9 + letters A-Z/a-z where cases are merged when visually similar.
#  Key challenge: digit/letter confusion — '0'↔'O', '1'↔'l'↔'I', '5'↔'S'.
#  NO horizontal flip — b/d, p/q, 6/9 confusions all apply here.

In [ ]:
# importing necessary dependencies
import os, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import math

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


DATASET      = "emnist/balanced"
NUM_CLASSES  = 47
IMG          = 28
BS           = 64
EPOCHS       = 100
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.1
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/balanced"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELS = (
    [str(d) for d in range(10)]
    + ['A','B','C','D','E','F','G','H','I','J','K','L','M',
       'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
       'a','b','d','e','f','g','h','n','q','r','t']
)  # 47 labels
os.makedirs(RESULTS_DIR, exist_ok=True)

#  DATA LOADING
#  EMNIST in torchvision uses transpose/flip to match the standard orientation.
#  torchvision applies the canonical fix automatically via the target_type arg.
print(f"[INFO] Loading EMNIST Balanced via torchvision ...")

train_transform = transforms.Compose([
    # EMNIST images are stored transposed; torchvision does NOT auto-fix this
    # for all splits — we handle it in the dataset wrapper below.
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),          # maps [0,1] → [-1,1]
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.25, contrast=0.25),
    ], p=0.8),
    transforms.Pad(2, fill=-1.0),                  # pad then crop (same as TF version)
    transforms.RandomCrop(IMG),
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])


class EMNISTFixed(datasets.EMNIST):
    """
    EMNIST images are stored 90° rotated + mirrored relative to the usual
    handwriting orientation.  We apply the canonical fix (transpose + h-flip)
    so letters look upright.  We do NOT apply a random h-flip as data
    augmentation (b/d, p/q, 6/9 confusions).
    """
    def __getitem__(self, index):
        img, target = super().__getitem__(index)
        # img is a PIL Image at this point (before transform)
        # Fix EMNIST orientation: rotate 90° CCW then flip horizontally
        img = transforms.functional.rotate(img, -90)
        img = transforms.functional.hflip(img)
        if self.transform is not None:
            img = self.transform(img)
        return img, target


# Note: torchvision's EMNIST returns PIL images before transform,
# so we subclass to apply the fix before the main transform pipeline.
# However transforms.ToTensor() must come first — restructure slightly.

class EMNISTBalanced(torch.utils.data.Dataset):
    """Wraps torchvision EMNIST with orientation fix applied before transforms."""
    def __init__(self, root, split="train", transform=None, download=True):
        self._base = datasets.EMNIST(
            root=root, split="balanced", train=(split == "train"),
            download=download, transform=None,
        )
        self.transform = transform

    def __len__(self):
        return len(self._base)

    def __getitem__(self, index):
        img, label = self._base[index]
        # Fix orientation BEFORE any tensor transforms
        img = transforms.functional.rotate(img, -90)
        img = transforms.functional.hflip(img)
        if self.transform:
            img = self.transform(img)
        return img, label


full_train = EMNISTBalanced("./data", split="train",  transform=None,   download=True)
test_set   = EMNISTBalanced("./data", split="test",   transform=None,   download=True)

total   = len(full_train)
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val

train_idx, val_idx = range(n_train), range(n_train, total)


class TransformSubset(torch.utils.data.Dataset):
    """Apply a transform to a Subset lazily."""
    def __init__(self, dataset, indices, transform):
        self.dataset   = dataset
        self.indices   = list(indices)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, label = self.dataset[self.indices[idx]]
        if self.transform:
            img = self.transform(img)
        return img, label


train_dataset = TransformSubset(full_train, train_idx, train_transform)
val_dataset   = TransformSubset(full_train, val_idx,   val_test_transform)

class TransformDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset   = dataset
        self.transform = transform
    def __len__(self):  return len(self.dataset)
    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        return self.transform(img), label

test_dataset = TransformDataset(test_set, val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BS, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_set):,}")
print(f"[INFO] Steps/epoch: {steps_per_epoch} | Device: {DEVICE}")

#  BUILDING BLOCKS

def gelu(x):
    return F.gelu(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.fc1  = nn.Linear(channels, max(1, channels // reduction))
        self.fc2  = nn.Linear(max(1, channels // reduction), channels)

    def forward(self, x):
        b, c, _, _ = x.shape
        attn = self.gap(x).view(b, c)
        attn = F.relu(self.fc1(attn))
        attn = torch.sigmoid(self.fc2(attn)).view(b, c, 1, 1)
        return x * attn


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return F.gelu(x + self.net(x))


class DenseResBlock(nn.Module):
    """
    Dense residual block with downsampling via depthwise-separable conv.
    Mirrors the TF dense_res_block function.
    """
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.project = None
        if in_ch != out_ch:
            self.project = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)

        # Fuse concatenated r1+r2+r3 (3*out_ch → out_ch)
        self.fuse = nn.Sequential(
            nn.Conv2d(out_ch * 3, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        # Depthwise-separable stride-2 downsampling
        self.dw = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                            groups=out_ch, bias=False)
        self.pw = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

    def forward(self, x):
        if self.project:
            x = F.gelu(self.project(x))
        r1 = self.r1(x)
        r2 = self.r2(r1)
        r3 = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = F.gelu(self.fuse(cat))
        out = F.gelu(self.pw(self.dw(out)))
        return out


class AdaptiveFilterCapsule(nn.Module):
    """
    Lightweight capsule routing.
    capsule_dim=32 — richer routing to separate confusable pairs like 0/O, 1/I/l.
    """
    def __init__(self, in_features, num_classes, capsule_dim=32):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_features, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        b = x.size(0)
        h = F.gelu(self.fc1(x))
        h = self.fc2(h).view(b, self.num_classes, self.capsule_dim)  # (B,K,D)
        # x_sliced: first capsule_dim features repeated for each class
        x_sliced = x[:, :self.capsule_dim].unsqueeze(1).expand(-1, self.num_classes, -1)
        caps = (x_sliced * h).sum(dim=-1)  # (B, K)
        return self.bn(caps)


#  MODEL
class WhatNetBalanced(nn.Module):
    def __init__(self, num_classes=47):
        super().__init__()
        K = num_classes

        # Stem
        self.stem_t = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32), nn.GELU())
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 16, (1, 5), padding=(0, 2), bias=False), nn.BatchNorm2d(16), nn.GELU())
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 16, (5, 1), padding=(2, 0), bias=False), nn.BatchNorm2d(16), nn.GELU())

        self.stem_ca   = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64), nn.GELU())

        # Encoder
        self.enc1 = DenseResBlock(64,  96)
        self.enc2 = DenseResBlock(96,  192)
        self.enc3 = DenseResBlock(192, 384)

        # Multi-scale GAP projection
        self.gap1_proj = nn.Sequential(nn.Linear(96,  128), nn.GELU())
        self.gap2_proj = nn.Sequential(nn.Linear(192, 128), nn.GELU())
        # enc3 outputs 384 features, concatenated with 128+128 = 640 total

        fused_dim = 128 + 128 + 384  # 640

        # Adaptive Filter Capsule
        self.afc = AdaptiveFilterCapsule(fused_dim, K, capsule_dim=32)

        # Dense head
        self.head_dense = nn.Linear(fused_dim, 256, bias=False)
        self.head_ln    = nn.LayerNorm(256)
        self.head_drop  = nn.Dropout(0.35)
        self.head_logits= nn.Linear(256, K)

        # Gated fusion
        self.gate = nn.Linear(K * 2, 2)

    def forward(self, x):
        # Stem
        t    = self.stem_t(x)
        h    = self.stem_h(x)
        v    = self.stem_v(x)
        stem = torch.cat([t, h, v], dim=1)          # (B, 64, 28, 28)
        stem = self.stem_ca(stem)
        stem = self.stem_proj(stem)

        # Encoder
        e1 = self.enc1(stem)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        # Multi-scale GAP fusion
        g1 = self.gap1_proj(e1.mean(dim=[2, 3]))    # (B, 128)
        g2 = self.gap2_proj(e2.mean(dim=[2, 3]))    # (B, 128)
        g3 = e3.mean(dim=[2, 3])                    # (B, 384)
        fused = torch.cat([g1, g2, g3], dim=1)      # (B, 640)

        # AFC
        afc_out = self.afc(fused)                   # (B, K)

        # Dense head
        x_h = self.head_dense(fused)
        x_h = self.head_ln(x_h)
        x_h = F.gelu(x_h)
        x_h = self.head_drop(x_h)
        x_h = self.head_logits(x_h)                 # (B, K)

        # Gated fusion
        combined = torch.cat([x_h, afc_out], dim=1) # (B, 2K)
        gate     = F.softmax(self.gate(combined), dim=-1)  # (B, 2)
        out      = x_h * gate[:, 0:1] + afc_out * gate[:, 1:2]
        return out


def build_model(num_classes=47):
    return WhatNetBalanced(num_classes).to(DEVICE)

#  LR SCHEDULE — Warmup + Cosine Decay
def warmup_cosine_schedule(optimizer, total_steps, warmup_steps):
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / max(warmup_steps, 1)
        progress = (current_step - warmup_steps) / max(total_steps - warmup_steps, 1)
        cosine   = max(0.5 * (1.0 + math.cos(math.pi * progress)), 1e-6 / LR)
        return cosine
    return LambdaLR(optimizer, lr_lambda)

#  LABEL-SMOOTHED CROSS ENTROPY
class LabelSmoothingCE(nn.Module):
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        self.smoothing    = smoothing
        self.num_classes  = num_classes
        self.log_softmax  = nn.LogSoftmax(dim=-1)

    def forward(self, logits, targets):
        log_prob = self.log_softmax(logits)
        # one-hot smooth targets
        with torch.no_grad():
            smooth_targets = torch.full_like(log_prob, self.smoothing / (self.num_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        loss = -(smooth_targets * log_prob).sum(dim=-1).mean()
        return loss

#  TRAIN LOOP
model     = build_model(NUM_CLASSES)
criterion = LabelSmoothingCE(NUM_CLASSES, LABEL_SMOOTH).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

warmup_steps = steps_per_epoch * WARMUP_EP
scheduler    = warmup_cosine_schedule(optimizer, total_steps, warmup_steps)

print(f"[INFO] Params: {sum(p.numel() for p in model.parameters()):,}")

best_val_acc  = 0.0
patience_ctr  = 0
PATIENCE      = 12
history       = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(imgs)
            loss   = criterion(logits, labels)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
            total_loss += loss.item() * imgs.size(0)
            preds       = logits.argmax(dim=1)
            correct    += (preds == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total


for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader,   train=False)

    history["loss"].append(tr_loss)
    history["accuracy"].append(tr_acc)
    history["val_loss"].append(va_loss)
    history["val_accuracy"].append(va_acc)

    print(f"Epoch {epoch:3d}/{EPOCHS}  "
          f"loss={tr_loss:.4f}  acc={tr_acc*100:.2f}%  "
          f"val_loss={va_loss:.4f}  val_acc={va_acc*100:.2f}%")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(model.state_dict(),
                   os.path.join(RESULTS_DIR, "best_model.pt"))
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"[INFO] Early stopping at epoch {epoch}")
            break

# Restore best weights
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "best_model.pt"),
                                 map_location=DEVICE))

#  EVALUATE
model.eval()
test_loss_total, test_correct, test_total = 0.0, 0, 0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        logits       = model(imgs)
        loss         = criterion(logits, labels)
        preds        = logits.argmax(dim=1)

        test_loss_total += loss.item() * imgs.size(0)
        test_correct    += (preds == labels).sum().item()
        test_total      += imgs.size(0)

        preds_np  = preds.cpu().numpy()
        labels_np = labels.cpu().numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds_np == c) & (labels_np == c))
            fp[c] += np.sum((preds_np == c) & (labels_np != c))
            fn[c] += np.sum((preds_np != c) & (labels_np == c))
            correct_per_class[c] += np.sum(preds_np[labels_np == c] == c)
            total_per_class[c]   += np.sum(labels_np == c)

test_loss = test_loss_total / test_total
test_acc  = test_correct / test_total * 100.0

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {sum(p.numel() for p in model.parameters()):,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BALANCED", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4),
    "params": sum(p.numel() for p in model.parameters()),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

[INFO] Loading EMNIST Balanced via torchvision ...


100%|██████████| 562M/562M [00:02<00:00, 214MB/s]  


[INFO] Train: 101,520 | Val: 11,280 | Test: 18,800
[INFO] Steps/epoch: 1586 | Device: cuda
[INFO] Params: 12,112,923
Epoch   1/100  loss=2.3769  acc=49.32%  val_loss=1.1643  val_acc=84.75%
Epoch   2/100  loss=1.2244  acc=83.62%  val_loss=1.1320  val_acc=85.27%
Epoch   3/100  loss=1.1446  acc=85.69%  val_loss=1.0582  val_acc=87.31%
Epoch   4/100  loss=1.0956  acc=86.68%  val_loss=1.0218  val_acc=87.91%
Epoch   5/100  loss=1.0581  acc=87.41%  val_loss=1.0011  val_acc=88.68%
Epoch   6/100  loss=1.0240  acc=88.04%  val_loss=1.0023  val_acc=88.33%
Epoch   7/100  loss=0.9970  acc=88.68%  val_loss=0.9716  val_acc=89.26%
Epoch   8/100  loss=0.9769  acc=89.24%  val_loss=0.9673  val_acc=89.28%
Epoch   9/100  loss=0.9659  acc=89.48%  val_loss=0.9699  val_acc=89.29%
Epoch  10/100  loss=0.9544  acc=89.82%  val_loss=0.9689  val_acc=89.10%
Epoch  11/100  loss=0.9457  acc=90.14%  val_loss=0.9520  val_acc=89.81%
Epoch  12/100  loss=0.9375  acc=90.28%  val_loss=0.9529  val_acc=89.50%
Epoch  13/100  loss